## Question 4 - Method of Relaxation in 2D

One of the major properties of a solution to Laplace's equation is that the value of the potential at a point is equal to the average of all the points surrounding it. We will exploit this using the 4 points that surround each point on a 2D grid.

Explicitly, this is equivalent to replacing the value of the potential $V_{i,j}$ with the arithmetic average of its nearest-neighbors on the mesh (i.e. nearest-neigbor averaging)
$$V_{i,j} = \frac{1}{4}(V_{i-1,j} + V_{i,j-1} + V_{i,j+1}+V_{i+1,j})$$



The procedure for solving Laplace's Eq. numerically involves, usually, the following steps:

1. Slice up the space where $\nabla^2 V = 0$ (and the boundary) into a mesh of points, that are spaced an equal distance apart.
2. Set the value of the boundary points given the specific problem you intend to solve. In this problem we start with $20 \text{V}$ on one edge and zero on the other 3 edges.
3. Starting at some location away from the boundary, systematically loop through each point applying the averaging function given above. *Typically, one can start at a corner and then go systematically across then down or vice versa.*
4. Compute the deviation between the starting values of the potential and the new values after a full iteration at each point. Compare this deviation to the tolerance decided upon before starting the calculation. We will use the maximum error as the deviation.
5. Repeat steps 3 and 4 until the deviation is below the tolerance. You should also build in a maximum number of steps, so that the program cannot run infinitely.
6. Plot results in a 3D plot (using $V(x,y) = z$ for 2D space, and a color gradient for $V(x,y,z)$)

### Task

Expand the 1D method of relaxation code to solve the 2D problem. Boundary conditions are 
$$x,y\in[0,10] ,\quad 
\begin{cases}
V(0,y) =& 20\text{V}\\
V(x,0) =& 0\\
V(10,y)=& 0\\
V(x,10)=& 0\end{cases}$$
Pick a reasonable, adjustable step size for the mesh.
Produce a 3D plot of the potential.

---

#### Solution

Let the region $\mathcal{D}$ where $\nabla^2 V = 0$ be composed of $N$ many squares.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display_latex, display
%matplotlib tk

In [ ]:
N=30

In [ ]:
x_bounds,y_bounds = ((0,10),(0,10))
x_broken = np.linspace(x_bounds[0],x_bounds[1],N+1)
y_broken = np.linspace(y_bounds[0],y_bounds[1],N+1)
scalars_init = (0,0)
display_latex(len(x_broken),len(scalars_init),raw=True)
#display(scalars_init)

In [ ]:
x, y = np.meshgrid(x_broken,y_broken,indexing='ij')
x_shape = x.shape
#is the same as y.shape


In [ ]:
scalar_field = np.broadcast_to(scalars_init,(*x_shape,2))
full_field = np.concatenate([x[...,None],y[...,None],scalar_field], axis=-1)


In [ ]:
full_field[0,:,2] = np.full(N+1,20)
#boundary condition, fulfilled cuz other sides are 0.

Let $|Err_{\text{max}}| = \varepsilon = 10^-9$.

In [ ]:
epsilon = 10e-9

def VMean(big_array:np.ndarray,i:int,j:int) -> np.float64:
    res = (1/4)*(big_array[i-1,j][2]+big_array[i+1,j][2]+big_array[i,j-1][2]+big_array[i,j+1][2])
    return res

def MaxErr(big_array:np.ndarray):
    max = np.max(big_array[:,:,3])
    return max


In [ ]:
err=2
oldarray = full_field.copy()
num = 0
max_num = 10000
fig= plt.figure(figsize=(8,6))
ax1 = fig.add_subplot(1,2,1,projection='3d')
ax2 = fig.add_subplot(1,2,2,projection='3d')
ax1.set_title("Relaxation solution to Laplace's Equation")
ax1.set_xlabel(r"$x$")
ax1.set_ylabel(r"$y$")
ax1.set_zlabel(r"$V(x,y)$")
ax2.set_xlabel(r"$x$")
ax2.set_ylabel(r"$y$")
ax2.set_zlabel(r"$Err(x,y)$")
while num <= max_num and err>=epsilon:
    newarray = oldarray.copy()
    for i in range(1,N-1):
        for j in range(1,N-1):
            #print(VMean(oldarray,i,j))
            #print(newarray[i][j][2])
            newarray[i,j][2] = VMean(oldarray,i,j)
            newarray[i,j][3] = newarray[i,j][2] - oldarray[i,j][2]
    newarray[:,:,3] = np.abs(newarray[:,:,3])
    err = MaxErr(newarray)
    oldarray=newarray
    num+=1
print(num)
print(err)

ax1.plot_surface(x,y,newarray[:,:,2],label='Voltage',cmap='magma')
ax2.set_title(f"Relaxation error over {num} iterations")
ax2.plot_surface(x,y,newarray[:,:,3],label='Error',cmap='inferno',axlim_clip=True)
ax2.set_zscale('symlog',linthresh=1e-12)
ax2.set_zlim3d((1e-11,2*err))
ax1.legend()
ax2.legend()
display(fig.canvas)